# Import Data and Libraries

In [ ]:
import hickle
import pandas as pd
import scipy.signal
# import shap
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import FormatStrFormatter
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
# from astropy.table import Table
from scipy import ndimage
from tqdm import tqdm
import corner
from PIL import Image
import io

import os

import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
import torchvision.transforms as T

from scipy.stats import chi2
from sklearn.datasets import make_s_curve
import matplotlib as mpl
import matplotlib.pyplot as plt
import tqdm

import ili
from ili.dataloaders import NumpyLoader
from ili.inference import InferenceRunner
from ili.validation.metrics import PosteriorCoverage, PlotSinglePosterior

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

from datasets import load_dataset, DatasetDict
from transformers import ViTFeatureExtractor, ViTForImageClassification, ViTModel, AdamW, AutoImageProcessor, AutoModel
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
from tqdm import tqdm

In [ ]:
GSWLC_data = load_dataset('juanpabloalfonzo/GSWLC_Data',split='train')
gswlc_data = GSWLC_data.to_pandas()

gswlc_images = load_dataset('juanpabloalfonzo/GSWLC_Images',split='train')
gswlc_images = gswlc_images['images'].copy()

# Set Up Data Loaders and Training Loops

In [ ]:
from sklearn.model_selection import train_test_split

class GalaxyDataset(Dataset):
    """Custom dataset for galaxy images and properties"""
    
    def __init__(self, images, labels, processor):
        """
        Args:
            images: numpy array of shape (N, 64, 64, 3)
            labels: numpy array of shape (N, 2) containing [log_mstar, log_sfr]
            processor: HuggingFace image processor
        """
        self.images = images
        self.labels = labels
        self.processor = processor
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Get image (H, W, C format)
        image = self.images[idx]
        image = Image.fromarray(image)
        image = np.array(image)
        
        # Ensure image is in uint8 format (0-255) if needed
        if image.max() <= 1.0:
            image = (image * 255).astype(np.uint8)
        
        # Process image using ViT processor
        # This will resize to 224x224 and normalize
        processed = self.processor(images=image, return_tensors="pt")
        pixel_values = processed['pixel_values'].squeeze(0)  # Remove batch dim
        
        # Get labels
        labels = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        return pixel_values, labels


class ViTForRegression(nn.Module):
    """ViT model with regression head for predicting continuous values"""
    
    def __init__(self, model_name, num_outputs=2, dropout=0.1):
        """
        Args:
            model_name: Name of pretrained ViT model
            num_outputs: Number of regression outputs (2 for log_mstar and log_sfr)
            dropout: Dropout rate for regularization
        """
        super().__init__()
        
        # Load pretrained ViT
        self.vit = AutoModel.from_pretrained(model_name)
        
        # Get hidden size from ViT config
        hidden_size = self.vit.config.hidden_size
        
        # Add regression head
        self.regressor = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_outputs)
        )
        
    def forward(self, pixel_values):
        # Get ViT outputs
        outputs = self.vit(pixel_values=pixel_values)
        
        # Use [CLS] token representation
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        # Pass through regression head
        predictions = self.regressor(cls_output)
        
        return predictions


def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    
    pbar = tqdm(dataloader, desc='Training')
    for pixel_values, labels in pbar:
        pixel_values = pixel_values.to(device)
        labels = labels.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        predictions = model(pixel_values)
        loss = criterion(predictions, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
    
    return total_loss / len(dataloader)


def validate(model, dataloader, criterion, device):
    """Validate the model"""
    model.eval()
    total_loss = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for pixel_values, labels in tqdm(dataloader, desc='Validating'):
            pixel_values = pixel_values.to(device)
            labels = labels.to(device)
            
            predictions = model(pixel_values)
            loss = criterion(predictions, labels)
            
            total_loss += loss.item()
            all_predictions.append(predictions.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    all_predictions = np.vstack(all_predictions)
    all_labels = np.vstack(all_labels)
    
    # Calculate per-property metrics
    mse_mstar = np.mean((all_predictions[:, 0] - all_labels[:, 0]) ** 2)
    mse_sfr = np.mean((all_predictions[:, 1] - all_labels[:, 1]) ** 2)
    
    return total_loss / len(dataloader), all_predictions, all_labels, mse_mstar, mse_sfr


def plot_results(predictions, labels, epoch, save_path='predictions_plot.png'):
    """Plot predicted vs true values"""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot log_mstar
    axes[0].scatter(labels[:, 0], predictions[:, 0], alpha=0.5, s=1)
    axes[0].plot([labels[:, 0].min(), labels[:, 0].max()], 
                 [labels[:, 0].min(), labels[:, 0].max()], 
                 'r--', lw=2, label='Perfect prediction')
    axes[0].set_xlabel('True log(M*)')
    axes[0].set_ylabel('Predicted log(M*)')
    axes[0].set_title(f'Stellar Mass - Epoch {epoch}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot log_sfr
    axes[1].scatter(labels[:, 1], predictions[:, 1], alpha=0.5, s=1)
    axes[1].plot([labels[:, 1].min(), labels[:, 1].max()], 
                 [labels[:, 1].min(), labels[:, 1].max()], 
                 'r--', lw=2, label='Perfect prediction')
    axes[1].set_xlabel('True log(SFR)')
    axes[1].set_ylabel('Predicted log(SFR)')
    axes[1].set_title(f'Star Formation Rate - Epoch {epoch}')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    # plt.close()
    plt.show()

model_name = "google/vit-base-patch16-224"
batch_size = 32
num_epochs = 50
learning_rate = 1e-4
weight_decay = 0.01
test_size = 0.2
val_size = 0.1


print("Loading data...")

images = []
for img_bytes in tqdm(np.array(gswlc_images_skyserver), desc='Loading images'):
    image = Image.open(io.BytesIO(img_bytes))
    images.append(np.array(image))
print(f"Images shape: {np.array(images).shape}")

images = np.array(images)

# Load labels

tempssfr = np.array(gswlc_data['logSSFR_50']).copy()
tempssfr[tempssfr < -3.5] = tempssfr[tempssfr < -3.5] + 0.3*np.random.normal(size=(np.sum(tempssfr < -3.5),))

log_mstar = gswlc_data['logM_50_x']
log_sfr = tempssfr - 9 


# Stack labels
labels = np.stack([log_mstar, log_sfr], axis=1)
print(f"Labels shape: {labels.shape}")

# Check for NaN values and remove them
valid_mask = ~(np.isnan(labels).any(axis=1))
images = images[valid_mask]
labels = labels[valid_mask]
print(f"After removing NaN: {images.shape[0]} samples")

# Print label statistics
print(f"\nLabel statistics:")
print(f"log_mstar - mean: {labels[:, 0].mean():.3f}, std: {labels[:, 0].std():.3f}")
print(f"log_sfr   - mean: {labels[:, 1].mean():.3f}, std: {labels[:, 1].std():.3f}")

# Split data: train, validation, test
X_temp, X_test, y_temp, y_test = train_test_split(
    images, labels, test_size=test_size, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_size/(1-test_size), random_state=42
)

print(f"\nData split:")
print(f"Train: {len(X_train)} samples")
print(f"Val:   {len(X_val)} samples")
print(f"Test:  {len(X_test)} samples")

# Initialize processor
print("\nInitializing model...")
processor = AutoImageProcessor.from_pretrained(model_name)

# Create datasets
train_dataset = GalaxyDataset(X_train, y_train, processor)
val_dataset = GalaxyDataset(X_val, y_val, processor)
test_dataset = GalaxyDataset(X_test, y_test, processor)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=4,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4,
    pin_memory=True
)
test_loader = DataLoader(
    test_dataset, 
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4,
    pin_memory=True
)

# Train the ViT

In [ ]:
# Initialize model
model = ViTForRegression(model_name, num_outputs=2, dropout=0.1)
model = model.to(device)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=learning_rate, 
    weight_decay=weight_decay
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=num_epochs
)

# Visualize Training Results on Test Set

In [ ]:
test_loss, test_preds, test_labels, test_mse_mstar, test_mse_sfr = validate(
    model, test_loader, criterion, device
)

print(f"\nTest Results:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test MSE - log_mstar: {test_mse_mstar:.4f}, log_sfr: {test_mse_sfr:.4f}")
print(f"Test RMSE - log_mstar: {np.sqrt(test_mse_mstar):.4f}, log_sfr: {np.sqrt(test_mse_sfr):.4f}")

# Calculate R² scores
r2_mstar = 1 - (np.sum((test_labels[:, 0] - test_preds[:, 0])**2) / 
                np.sum((test_labels[:, 0] - test_labels[:, 0].mean())**2))
r2_sfr = 1 - (np.sum((test_labels[:, 1] - test_preds[:, 1])**2) / 
              np.sum((test_labels[:, 1] - test_labels[:, 1].mean())**2))

print(f"Test R² - log_mstar: {r2_mstar:.4f}, log_sfr: {r2_sfr:.4f}")

# Plot final test results
plot_results(test_preds, test_labels, 'Test', 'fine_tuned_vit_test_predictions_manga_matched.png')

# Extract Image Embeddings

In [ ]:
# 1. Define a dataset with the FULL data arrays (images and labels)
# Note: We use the original 'images' and 'labels' variables from before the split
full_dataset = GalaxyDataset(images, labels, processor)

# 2. Create a DataLoader
# CRITICAL: shuffle=False ensures embedding[0] corresponds to images[0]
full_loader = DataLoader(
    full_dataset, 
    batch_size=32,  # You can use a larger batch size for inference (e.g., 64 or 128) to speed it up
    shuffle=False, 
    num_workers=4,
    pin_memory=True
)

# Setup Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViTForRegression(model_name="google/vit-base-patch16-224")
model.load_state_dict(torch.load('vit_finetuned.pt', map_location=device))
model.to(device)
model.eval()

# Storage
full_embeddings = []

print(f"Extracting embeddings for {len(full_dataset)} galaxies...")

with torch.no_grad():
    for pixel_values, _ in tqdm(full_loader, desc="Processing Full Dataset"):
        pixel_values = pixel_values.to(device)
        
        # Pass through the ViT backbone only
        outputs = model.vit(pixel_values=pixel_values)
        
        # Extract [CLS] token (index 0)
        # Shape: [Batch_Size, 768]
        cls_output = outputs.last_hidden_state[:, 0, :]
        
        full_embeddings.append(cls_output.cpu().numpy())

# Stack into one large array
# Final Shape: (Total_Galaxies, 768)
final_embedding_matrix = np.vstack(full_embeddings)

print(f"Done. Final shape: {final_embedding_matrix.shape}")

finetuned_embeds = final_embedding_matrix